# state-dict-load — ex1: load a checkpoint into a head-swapped model with strict=False

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `state-dict-load`. Running the final beacon cell reports progress against the `Transfer: state_dict load` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: state_dict load` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`state-dict-load`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "state-dict-load"
DD_SUBTOPIC = "Transfer: state_dict load"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `load_state_dict` + `strict=False` — quick refresher

A `state_dict` is an ordered dict mapping parameter / buffer names to tensors. Loading it onto a model:

```
result = model.load_state_dict(t.load('checkpoint.pt'), strict=True)
```

**`strict=True` (the default)** — every key in the checkpoint must exist on the model AND vice versa, with shape-compatible tensors. Any mismatch raises `RuntimeError`.

**`strict=False`** — load whatever matches, return an `_IncompatibleKeys(missing_keys, unexpected_keys)` named tuple listing the discrepancies. Use this when:
- You've swapped the head — old `fc.weight/bias` becomes **unexpected**; new `fc.weight/bias` becomes **missing**.
- You're loading a smaller checkpoint into a bigger model (some layers stay at init).
- You're cross-loading between subtly-different architectures.

**The return value.**
```
result = model.load_state_dict(ckpt, strict=False)
print(result.missing_keys)     # keys model has, ckpt didn't provide
print(result.unexpected_keys)  # keys ckpt has, model doesn't want
```

**Shape mismatches are NOT silently fixed by `strict=False`.** If a key exists in both with INCOMPATIBLE shapes, it still raises (though as of recent PyTorch versions, this is surfaced via `assign=False` mode warnings, not an immediate hard error in all code paths). The safe pattern is: pop the mismatching keys from the state_dict BEFORE calling `load_state_dict`.

**Why this is the transfer-learning workflow.** Pretrained-checkpoint + swapped-head means: missing = `['fc.weight', 'fc.bias']` (head was replaced AFTER load, or load skipped them), unexpected = same names from the old 1000-class head. `strict=False` says 'I know about this, move on'.

### Exercise 1 — load a checkpoint into a head-swapped model with strict=False

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the `_IncompatibleKeys(missing, unexpected)` return value of `load_state_dict(strict=False)` after swapping a model's head, distinguishing missing keys (model has, ckpt didn't provide) from unexpected (ckpt has, model doesn't want).
> Keywords: load_state_dict, strict-false, incompatible-keys, missing, unexpected
> ```

**KCs targeted:** `load-state-dict-strict-false`, `interpret-incompatible-keys-tuple`

A toy model + simulated checkpoint are provided:

```
class ToyModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = nn.Linear(10, 16)
        self.fc = nn.Linear(16, n_classes)
    def forward(self, x):
        return self.fc(self.backbone(x))
```

Implement `ex1_load_with_head_swap(model, checkpoint)` that:

1. Calls `model.load_state_dict(checkpoint, strict=False)`.
2. Returns a 3-tuple `(model, missing, unexpected)`:
   - `model`: the loaded model (which the test then runs forward on).
   - `missing`: a Python **list** of strings — the `missing_keys` from the returned `_IncompatibleKeys` named-tuple. These are state-dict keys the model has but the checkpoint didn't provide.
   - `unexpected`: a Python **list** of strings — the `unexpected_keys` from the returned tuple. These are checkpoint keys the model didn't want.

**Typical scenario for the test.** The checkpoint was saved from a 1000-class model (`fc.weight: (1000, 16), fc.bias: (1000,)`), but the user is loading it into a 7-class model (`fc.weight: (7, 16), fc.bias: (7,)`). The shape mismatch on `fc.*` keys is what makes `strict=True` raise — so `strict=False` is the right escape hatch, and the checkpoint is pre-cleaned by dropping the mismatching `fc.*` keys.

**Reading the result.**
- `backbone.weight, backbone.bias` → present in both → loaded successfully.
- `fc.weight, fc.bias` → present in model only (checkpoint dropped them) → appear in `missing`.
- A leftover `extra_buffer` key → present in checkpoint only → appears in `unexpected`.

**Don't pre-process the checkpoint** — just pass it to `load_state_dict` with `strict=False`. The test constructs the checkpoint to produce the expected missing/unexpected lists.

In [ ]:
import torch.nn as nn

class ToyModel(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = nn.Linear(10, 16)
        self.fc = nn.Linear(16, n_classes)
    def forward(self, x):
        return self.fc(self.backbone(x))


def ex1_load_with_head_swap(model, checkpoint):
    """Load checkpoint with strict=False; return (model, missing_list, unexpected_list)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    # Build a 7-class model.
    model = ToyModel(n_classes=7)

    # Simulate a 'pretrained' checkpoint:
    #  - has backbone.weight, backbone.bias (shape matches → load).
    #  - missing fc.weight, fc.bias entirely (head was stripped before save).
    #  - has an extra 'extra_buffer' key the model doesn't want.
    rng = t.Generator().manual_seed(0)
    checkpoint = {
        'backbone.weight': t.randn(16, 10, generator=rng),
        'backbone.bias':   t.randn(16,     generator=rng),
        'extra_buffer':    t.zeros(5),
    }

    # Snapshot backbone weights before load to confirm they got overwritten.
    backbone_w_target = checkpoint['backbone.weight'].clone()
    fc_w_before       = model.fc.weight.detach().clone()

    loaded_model, missing, unexpected = ex1_load_with_head_swap(model, checkpoint)

    # Backbone got the checkpoint values (load succeeded for matching keys).
    assert t.allclose(loaded_model.backbone.weight, backbone_w_target), (
        'backbone.weight should have been loaded from checkpoint'
    )

    # fc was NOT overwritten — checkpoint didn't provide it.
    assert t.equal(loaded_model.fc.weight, fc_w_before), (
        'fc.weight should be unchanged (checkpoint had no fc.* keys)'
    )

    # Missing keys: fc.weight and fc.bias.
    assert isinstance(missing, list), f'missing must be a list, got {type(missing).__name__}'
    assert set(missing) == {'fc.weight', 'fc.bias'}, (
        f'expected missing={{fc.weight, fc.bias}}, got {set(missing)}'
    )

    # Unexpected keys: extra_buffer.
    assert isinstance(unexpected, list), f'unexpected must be a list, got {type(unexpected).__name__}'
    assert set(unexpected) == {'extra_buffer'}, (
        f'expected unexpected={{extra_buffer}}, got {set(unexpected)}'
    )

    # Forward pass works — backbone was loaded, fc is at fresh-init.
    x = t.randn(2, 10)
    y = loaded_model(x)
    assert y.shape == (2, 7), f'forward output shape wrong: {tuple(y.shape)}'

    # Second scenario: NO missing, NO unexpected (clean reload of the same model).
    m_clean = ToyModel(n_classes=7)
    sd_full = m_clean.state_dict()  # has fc.* matching shapes
    m_target = ToyModel(n_classes=7)
    m_target, miss2, unexp2 = ex1_load_with_head_swap(m_target, sd_full)
    assert miss2 == [],  f'clean reload should have no missing, got {miss2}'
    assert unexp2 == [], f'clean reload should have no unexpected, got {unexp2}'

    # Third scenario: ONLY unexpected (checkpoint has extra keys, model has everything else).
    ckpt_extra = dict(sd_full)
    ckpt_extra['orphan_buffer'] = t.zeros(3)
    m3, miss3, unexp3 = ex1_load_with_head_swap(ToyModel(n_classes=7), ckpt_extra)
    assert miss3 == []
    assert set(unexp3) == {'orphan_buffer'}
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_load_with_head_swap(model, checkpoint):
    result = model.load_state_dict(checkpoint, strict=False)
    return model, list(result.missing_keys), list(result.unexpected_keys)
```

**Reading `_IncompatibleKeys`.** It's a `typing.NamedTuple` with two fields, BOTH lists of strings:
- `missing_keys`: keys the model expects but the checkpoint didn't provide. The corresponding params/buffers stay at their fresh-init values.
- `unexpected_keys`: keys the checkpoint provided but the model doesn't want. They're silently dropped.

**Why `strict=True` is the default.** Loose loading is DANGEROUS — a typo in a key name silently leaves a param at random init and you get a model that 'loaded' but behaves like a random net. `strict=True` forces every discrepancy to surface immediately. Use `strict=False` only when you EXPECT mismatches (transfer learning, architectural surgery, partial loads).

**Mismatch in shape ≠ missing/unexpected.** If a key exists in both with INCOMPATIBLE shapes, modern PyTorch surfaces this with a `RuntimeError` even under `strict=False`. The safe pre-clean step before transfer learning:
```
for k in list(checkpoint.keys()):
    if k.startswith('fc.'):
        del checkpoint[k]      # drop head — will reappear in missing
```
Now `strict=False` happily loads the rest.

**Composes with the other transfer-learning atoms.** Typical full recipe:
1. Build model with new `n_classes` (or swap head — `replace-final-head` atom).
2. Strip head keys from checkpoint.
3. `load_state_dict(checkpoint, strict=False)` — this atom.
4. Freeze backbone — `freeze-requires-grad` atom.
5. Train head only.

All four atoms in this batch (`replace-final-head`, `state-dict-load`, plus `register-buffer` for understanding what's saved, plus `freeze-requires-grad` from batch-4) compose into the canonical transfer-learning workflow.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()